In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path.cwd().parent / 'data' / 'interim'
ONBOARDING_WINDOW = 90   # observation window in days
BUFFER = 30              # extra days to confirm churn
DATA_END = pd.Timestamp('2032-01-14', tz='UTC')


In [ ]:
# Load NB01 outputs
new_clients  = pd.read_csv(DATA / 'new_clients.csv')
tx_with_reg  = pd.read_csv(DATA / 'tx_with_reg.csv')
new_clients['registration_date'] = pd.to_datetime(new_clients['registration_date'], utc=True)
tx_with_reg['date']              = pd.to_datetime(tx_with_reg['date'], utc=True)
tx_with_reg['registration_date'] = pd.to_datetime(tx_with_reg['registration_date'], utc=True)
print(f'new_clients: {len(new_clients):,} | tx_with_reg: {len(tx_with_reg):,}')


In [ ]:
# Days from registration to each transaction
tx_with_reg['days_since_reg'] = (
    (tx_with_reg['date'] - tx_with_reg['registration_date'])
    .dt.total_seconds() / 86400
)

# First transaction per client
first_tx = (
    tx_with_reg.sort_values('date')
    .groupby('client_id')['days_since_reg'].min()
    .reset_index()
    .rename(columns={'days_since_reg': 'days_since_first_tx'})
)
print(f'Clients with any transaction: {len(first_tx):,}')


In [ ]:
# Eligible: first tx in days 0..90, enough buffer before DATA_END
eligible_ids = first_tx[
    (first_tx['days_since_first_tx'] >= 0) &
    (first_tx['days_since_first_tx'] <= ONBOARDING_WINDOW)
]['client_id'].unique()

eligible = new_clients[new_clients['client_id'].isin(eligible_ids)].copy()
eligible = eligible.merge(first_tx, on='client_id', how='left')

# Drop clients whose buffer window extends past dataset end
eligible['cutoff'] = eligible['registration_date'] + pd.Timedelta(days=ONBOARDING_WINDOW + BUFFER)
eligible = eligible[eligible['cutoff'] <= DATA_END].copy()
print(f'Eligible clients: {len(eligible):,}')


In [ ]:
# CHURNED=1 if no transaction in days (90, 120] after first tx
active_after = tx_with_reg[
    tx_with_reg['client_id'].isin(eligible['client_id'])
].copy()
active_after = active_after.merge(
    eligible[['client_id', 'days_since_first_tx']], on='client_id', how='left'
)
active_after['days_from_first'] = active_after['days_since_reg'] - active_after['days_since_first_tx']
returners = active_after[
    (active_after['days_from_first'] > ONBOARDING_WINDOW) &
    (active_after['days_from_first'] <= ONBOARDING_WINDOW + BUFFER)
]['client_id'].unique()

eligible['churned'] = (~eligible['client_id'].isin(returners)).astype(int)
print(f'CHURNED  (1): {eligible["churned"].sum():,}  ({eligible["churned"].mean()*100:.1f}%)')
print(f'RETAINED (0): {(eligible["churned"]==0).sum():,}  ({(eligible["churned"]==0).mean()*100:.1f}%)')


In [ ]:
# Figure: client type breakdown (thesis fig: client_type_breakdown.png)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

FIGURES = Path.cwd().parent / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

_first_tx = (
    tx_with_reg.groupby('client_id')['days_since_reg'].min()
    .reset_index().rename(columns={'days_since_reg': 'day_first_tx'})
)
_tx = tx_with_reg.merge(_first_tx, on='client_id', how='left')
_tx['days_since_first_tx'] = _tx['days_since_reg'] - _tx['day_first_tx']
_tx_ob = _tx[
    (_tx['days_since_first_tx'] >= 0) &
    (_tx['days_since_first_tx'] <= ONBOARDING_WINDOW) &
    (_tx['client_id'].isin(eligible['client_id']))
]
_tx_cnt = (_tx_ob.groupby('client_id')['transaction_id'].count()
           .reset_index().rename(columns={'transaction_id': 'tx_count'}))

_df = eligible.drop_duplicates(subset='client_id', keep='first').merge(_tx_cnt, on='client_id', how='left')
_df['tx_count'] = _df['tx_count'].fillna(0).astype(int)

n_total   = len(_df)
n_ret     = (_df['churned'] == 0).sum()
n_fading  = ((_df['churned'] == 1) & (_df['tx_count'] > 1)).sum()
n_onetime = ((_df['churned'] == 1) & (_df['tx_count'] <= 1)).sum()
pct_r = n_ret     / n_total * 100
pct_f = n_fading  / n_total * 100
pct_o = n_onetime / n_total * 100

fig, ax = plt.subplots(figsize=(8, 3.4))
left = 0
for pct, color, line1, line2 in [
    (pct_r, '#2e7d32', 'Retained',          f'{pct_r:.1f}%  ({n_ret:,})'),
    (pct_f, '#e65100', 'Fading (tx > 1)',   f'{pct_f:.1f}%  ({n_fading:,})'),
    (pct_o, '#b71c1c', 'One-time (tx = 1)', f'{pct_o:.1f}%  ({n_onetime:,})'),
]:
    ax.barh(0, pct, left=left, color=color, edgecolor='white', height=0.7)
    ax.text(left + pct/2,  0.12, line1, ha='center', va='center',
            color='white', fontweight='bold', fontsize=9)
    ax.text(left + pct/2, -0.18, line2, ha='center', va='center',
            color='white', fontsize=8.5)
    left += pct

ax.set_xlim(0, 100)
ax.set_ylim(-0.55, 0.55)
ax.set_xlabel('Share of analytical sample (%)', fontsize=9)
ax.set_yticks([])
ax.set_title(f'Client type breakdown — analytical sample (n = {n_total:,})',
             fontsize=11, fontweight='bold')
legend_handles = [
    mpatches.Patch(color='#2e7d32', label=f'Retained ({pct_r:.1f}%)'),
    mpatches.Patch(color='#e65100', label=f'Fading churners ({pct_f:.1f}%)'),
    mpatches.Patch(color='#b71c1c', label=f'One-time churners ({pct_o:.1f}%)'),
]
ax.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, -0.52),
          ncol=3, fontsize=9, frameon=True)
plt.tight_layout()
plt.savefig(FIGURES / 'client_type_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'n={n_total:,}  Retained={n_ret:,}  Fading={n_fading:,}  One-time={n_onetime:,}')


In [ ]:
# Cohort stability: churn rate by registration month
eligible['reg_month'] = eligible['registration_date'].dt.to_period('M')
cohort = eligible.groupby('reg_month')['churned'].agg(['mean', 'count'])
cohort['mean'] = cohort['mean'] * 100
print(cohort[cohort['count'] > 500].to_string())


In [ ]:
# Save labelled dataset
eligible.to_csv(DATA / 'eligible_with_labels.csv', index=False)
print(f'Saved eligible_with_labels.csv ({len(eligible):,} rows)')
print(f'Churn rate: {eligible["churned"].mean()*100:.1f}%')
